# Airbnb Pricing & Market Intelligence

## Data Modeling

This notebook prepares analysis-ready tables optimized for business analysis and Power BI reporting.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

listings = pd.read_csv(
    "../data/processed/listings_clean.csv",
    low_memory=False
)

reviews = pd.read_csv(
    "../data/processed/reviews_clean.csv",
    low_memory=False
)

print("Listings Shape :", listings.shape)
print("Reviews Shape  :", reviews.shape)

Listings Shape : (279434, 32)
Reviews Shape  : (5372983, 4)


### Source Data Assessment

In [2]:
print("LISTINGS COLUMNS")
print("-" * 50)
print(listings.columns.tolist())

print("\n")

print("REVIEWS COLUMNS")
print("-" * 50)
print(reviews.columns.tolist())

LISTINGS COLUMNS
--------------------------------------------------
['listing_id', 'name', 'host_id', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'city', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable']


REVIEWS COLUMNS
--------------------------------------------------
['listing_id', 'review_id', 'date', 'reviewer_id']


### Modeling Strategy

This project follows a simplified star-schema design.

- dim_listing serves as the primary analytical table containing listing, pricing, property, and location attributes.
- dim_host contains host-level intelligence and performance metrics.
- fact_reviews stores review transactions used for trend and seasonality analysis.

The model is designed to support market comparison, pricing intelligence, value-for-travel analysis, review trend analysis, and Power BI reporting.

### Create dim_listing

In [3]:
dim_listing = listings[
    [
        "listing_id",
        "host_id",
        "city",
        "neighbourhood",
        "property_type",
        "room_type",
        "accommodates",
        "bedrooms",
        "price",
        "minimum_nights",
        "maximum_nights",
        "instant_bookable",
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value"
    ]
].copy()

print("dim_listing Shape:")
print(dim_listing.shape)

dim_listing Shape:
(279434, 19)


In [4]:
print("Rows:")
print(len(dim_listing))

print("\nUnique listing_id:")
print(dim_listing["listing_id"].nunique())

Rows:
279434

Unique listing_id:
279434


### Save dim_listing

In [5]:
dim_listing.to_csv(
    "../data/processed/dim_listing.csv",
    index=False
)

print("dim_listing saved successfully.")

dim_listing saved successfully.


### Create dim_host

In [6]:
dim_host = listings[
    [
        "host_id",
        "host_since",
        "host_location",
        "host_response_time",
        "host_response_rate",
        "host_acceptance_rate",
        "host_is_superhost",
        "host_total_listings_count",
        "host_has_profile_pic",
        "host_identity_verified"
    ]
].drop_duplicates().copy()

print("dim_host Shape:")
print(dim_host.shape)

dim_host Shape:
(181857, 10)


In [7]:
print("Rows:")
print(len(dim_host))

print("\nUnique host_id:")
print(dim_host["host_id"].nunique())

Rows:
181857

Unique host_id:
181805


In [8]:
host_duplicates = dim_host[
    dim_host["host_id"].duplicated(keep=False)
].sort_values("host_id")

print("Affected Hosts:")
print(host_duplicates["host_id"].nunique())

host_duplicates.head(20)

Affected Hosts:
50


,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified
238058,290662,2010-11-18,"New York, New York, United States",within an hour,1.00,0.71,t,2.0,t,f
241128,290662,2010-11-18,"New York, New York, United States",within an hour,1.00,0.83,t,2.0,t,f
189141,357117,2011-01-22,"New York, New York, United States",within a day,0.80,0.80,f,1.0,t,t
86288,357117,2011-01-22,"New York, New York, United States",within a few hours,0.70,0.75,f,1.0,t,t
53042,510954,2011-04-15,"Rome, Lazio, Italy",NaN,NaN,NaN,f,6.0,t,t
237123,510954,2011-04-15,"Rome, Lazio, Italy",within a few hours,1.00,NaN,f,6.0,t,t
235142,761827,2011-06-30,"Rome, Lazio, Italy",within a few hours,1.00,0.89,f,30.0,t,t
239603,761827,2011-06-30,"Rome, Lazio, Italy",within a few hours,1.00,0.88,f,30.0,t,t
239705,767876,2011-07-01,"London, England, United Kingdom",within an hour,1.00,0.92,f,10.0,t,t
182264,767876,2011-07-01,"London, England, United Kingdom",within an hour,1.00,1.00,f,10.0,t,t


In [9]:
# Keep One Record Per Host

dim_host = (
    listings
    .sort_values(
        by=[
            "host_response_rate",
            "host_acceptance_rate"
        ],
        ascending=False,
        na_position="last"
    )
    .drop_duplicates(
        subset="host_id",
        keep="first"
    )
    [
        [
            "host_id",
            "host_since",
            "host_location",
            "host_response_time",
            "host_response_rate",
            "host_acceptance_rate",
            "host_is_superhost",
            "host_total_listings_count",
            "host_has_profile_pic",
            "host_identity_verified"
        ]
    ]
    .copy()
)

print("Rows:", len(dim_host))
print("Unique Hosts:", dim_host["host_id"].nunique())

Rows: 181805
Unique Hosts: 181805


In [10]:
dim_host.to_csv(
    "../data/processed/dim_host.csv",
    index=False
)

print("dim_host saved successfully.")

dim_host saved successfully.


### Create fact_reviews

In [11]:
fact_reviews = reviews[
    [
        "review_id",
        "listing_id",
        "reviewer_id",
        "date"
    ]
].copy()

print("Rows:")
print(len(fact_reviews))

print("\nUnique review_id:")
print(fact_reviews["review_id"].nunique())

print("\nDuplicate review_id:")
print(fact_reviews["review_id"].duplicated().sum())

Rows:
5372983

Unique review_id:
5372983

Duplicate review_id:
0


In [12]:
print("Unique Listing IDs:")
print(fact_reviews["listing_id"].nunique())

print("\nDate Range:")
print(fact_reviews["date"].min())

print(fact_reviews["date"].max())

Unique Listing IDs:
193556

Date Range:
2008-11-16
2021-03-01


### Save fact_reviews

In [13]:
fact_reviews.to_csv(
    "../data/processed/fact_reviews.csv",
    index=False
)

print("fact_reviews saved successfully.")

fact_reviews saved successfully.


### Modeling Summary

In [14]:
model_summary = pd.DataFrame({
    "Table": [
        "dim_listing",
        "dim_host",
        "fact_reviews"
    ],
    "Rows": [
        len(dim_listing),
        len(dim_host),
        len(fact_reviews)
    ]
})

print(model_summary.to_string(index=False))

       Table    Rows
 dim_listing  279434
    dim_host  181805
fact_reviews 5372983


## Modeling Conclusion

The Airbnb dataset was successfully transformed into a simplified analytical data model consisting of:

- dim_listing
- dim_host
- fact_reviews

The model separates listing, host and review information to support efficient analysis across pricing, market structure, customer demand, review behavior, and host performance.

Duplicate review assignments identified during data cleaning were removed prior to modeling, restoring review_id uniqueness within the fact table.

The resulting model provides a reliable foundation for SQL analysis, KPI development, and Power BI dashboard reporting.